# Impedance ML Engineering — Session 09
## Train a small neural network end to end
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
Can an MLP learn a useful telemetry predictor, and can you audit every stage of training?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Prepare neural inputs using training-only preprocessing.
- Build a Dataset/DataLoader and a small nn.Module.
- Write training and evaluation loops with correct modes.
- Compare validation results with ridge and persistence.

## R-to-Python / prerequisite bridge
Formula fit objects → explicit `nn.Module`, optimizer, and loop; training/evaluation are separate; dataloaders batch examples, not independent patients.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
The training loop owns parameter updates; the evaluation loop only computes predictions/metrics. `model.train()` and `.eval()` control behavior of layers such as dropout; `torch.no_grad()` controls gradient recording. They address different concerns. `DataLoader` batching changes computation order after the patient split, not the statistical unit of evaluation.

Adam maintains gradient-based optimizer state. A fresh model without a fresh optimizer is not a complete restart. Accumulate batch loss weighted by row count so a short final batch does not dominate the epoch mean.

Imports: `import torch`, `from torch import nn`, `from torch.utils.data import TensorDataset, DataLoader`. In a batch step: clear gradients → forward → loss → backward → optimizer step. For logged scalars use `.item()`; for sklearn metrics use `.detach().cpu().numpy()`.

## Task 1 — Prepare encoded inputs (7 min)
Build the same numeric/categorical preprocessor as Session 02, fit training only, transform train/validation to float32. Convert targets to (N, 1). Predict encoded width from fitted feature names. Keep a saved list of names/order; do not put patient IDs into tensors.

## Task 2 — Create loaders and check a batch (6 min)
Use `TensorDataset(torch.as_tensor(X_train),torch.as_tensor(y_train))` and `DataLoader(batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(42))`. Validation loader uses shuffle=False. Check matching row counts and inspect batch shapes. Shuffling training batches is allowed after patients have been partitioned.

## Task 3 — Define an MLP and initialize it (7 min)
Set seed 42; define an nn.Module with Linear(input_width, 16), ReLU, Linear(16, 1). Implement `forward(self,x)` and count trainable parameters. Check a forward pass. Hint: inherit `nn.Module`, call `super().__init__`, define layers in `__init__`; expose one output per row.

## Task 4 — Write one training epoch (10 min)
Use Adam(lr=0.01) and MSELoss. For each batch call zero_grad, forward, loss.backward, optimizer.step. Set model.train(). Log a row-weighted epoch MSE: multiply batch loss by batch size before summing. Acceptance: finite loss and changed weights. Do not average a small final batch equally with a full batch.

## Task 5 — Write a validation function (8 min)
Set model.eval() and use `with torch.no_grad():`. Concatenate predictions in loader order and return MSE plus MAE. Assert predictions and targets have exactly the same shape; convert detached CPU outputs to NumPy for sklearn metrics. Explain why validation must not update weights or optimizer state.

## Task 6 — Train a short fixed experiment (8 min)
Run 40 epochs maximum, recording training and validation loss. For this first loop report the fixed final epoch; checkpoint selection is Session 10. Plot curves. Check reproducibility from a fresh seed and fresh optimizer; do not call continued training a restart.

## Task 7 — Compare with simple models (5 min)
Fit summary ridge on training only, and score persistence/ridge/MLP on identical validation rows with MAE and patient-average MAE. Record parameter count and CPU time. Explain why a neural model need not beat ridge in this small dataset.

## Task 8 — Audit an AI training suggestion (1 min)
Explain the faults in `model.eval(); loss.backward(); optimizer.step()` used during validation and in reshaping only predictions before loss. Write three assertions you would demand before trusting an AI-generated training loop.

## Retrieval practice
Reconstruct optimizer zero_grad → forward → loss → backward → step from memory and explain where gradients live.

## Professional reasoning
1. Why is torch.no_grad separate from model.eval?
2. What could go wrong when targets and predictions have matching lengths but different row order?
3. How would an unseen categorical level be handled at inference?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Repeat with seeds 42, 43, 44 while keeping patient partitions fixed. Compare optimization variability without tuning seeds to choose the best result.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)